# 06 - Verify and publish the complete exact-v3 reproduction

Reads only outputs from the current fresh run folder, asserts the frozen v3 targets, creates a complete DEV-34 result package, and publishes that package to GitHub under `results/<run_id>/`.

Published Git artifacts include:
- comparative metric table (CSV + Markdown)
- normalized DEV-34 predictions for every reproduced model
- confusion matrices (CSV + PNG)
- classification reports (CSV + JSON)
- source metric/audit JSON files
- run provenance and a human-readable README

Large checkpoints, ComParE16 arrays and intermediate features remain on Drive. TEST is never accessed.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, shutil, subprocess, base64

DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
EXP_ROOT = DAIC / 'experiments' / 'exact_v3_reproduction'
ACTIVE = EXP_ROOT / 'ACTIVE_RUN.txt'
assert ACTIVE.is_file(), 'Run notebooks/00_start_new_run.ipynb first.'
RUN = EXP_ROOT / ACTIVE.read_text().strip()
assert RUN.is_dir(), RUN

REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
CORE_COMMIT = '29087f39a46ddf756b6a082ff7148cf90e336ca7'
CODE = Path('/content/reliability-aware-depression-kd-exact-v3')

token = userdata.get('GITHUB_TOKEN')
assert token, 'Add GITHUB_TOKEN to Colab Secrets.'
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy()
env['GIT_CONFIG_COUNT']='1'
env['GIT_CONFIG_KEY_0']='http.https://github.com/.extraheader'
env['GIT_CONFIG_VALUE_0']=f'AUTHORIZATION: basic {auth}'

if CODE.exists(): shutil.rmtree(CODE)
CODE.mkdir(parents=True)
subprocess.run(['git','-C',str(CODE),'init'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'remote','add','origin',f'https://github.com/{REPO}.git'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',CORE_COMMIT],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'checkout','--detach','FETCH_HEAD'],check=True,env=env)
head=subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()
assert head==CORE_COMMIT,(head,CORE_COMMIT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements.txt')],check=True)
print('RUN:',RUN)
print('Pinned core:',head)


In [ ]:
import pandas as pd, numpy as np, math
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

TEXT=RUN/'03_student_text'
AUDIO_BRANCH=RUN/'03_student_audio'
NOKD=RUN/'04_no_kd'
KD=RUN/'05_standard_kd'
TTEACH=RUN/'01_text_teacher'
ATEACH=RUN/'02_audio_teacher'/'dev_audit'

for p in [TEXT,AUDIO_BRANCH,NOKD,KD,TTEACH,ATEACH]:
    assert p.is_dir(), f'Missing required output: {p}'

# ---- Normalize all DEV-34 predictions into the same schema ----
preds={}

d=pd.read_csv(TTEACH/'dev_text_predictions.csv')
preds['IDIAP text teacher']=pd.DataFrame({
    'participant_id':d['participant_id'].astype(int),
    'label':d['label'].astype(int),
    'probability':d['text_probability'].astype(float),
    'prediction':d['text_prediction'].astype(int),
})

d=pd.read_csv(ATEACH/'dev_participant_predictions.csv')
preds['USSD audio teacher']=pd.DataFrame({
    'participant_id':d['participant_id'].astype(int),
    'label':d['label'].astype(int),
    'probability':d['kd_probability'].astype(float),      # soft probability for AUROC
    'prediction':d['author_prediction'].astype(int),     # author majority vote for hard metrics
})

z=np.load(TEXT/'dev_text_embeddings.npz')
preds['v3 text branch']=pd.DataFrame({
    'participant_id':z['participant_ids'].astype(int),
    'label':z['labels'].astype(int),
    'probability':z['probability'].astype(float),
})
preds['v3 text branch']['prediction']=(preds['v3 text branch']['probability']>=0.5).astype(int)

for name,path in [('No-KD multimodal',NOKD/'dev_predictions.csv'),
                  ('Standard KD',KD/'dev_predictions.csv')]:
    d=pd.read_csv(path)
    preds[name]=pd.DataFrame({
        'participant_id':d['participant_id'].astype(int),
        'label':d['label'].astype(int),
        'probability':d['probability'].astype(float),
        'prediction':d['prediction'].astype(int),
    })

for name,d in preds.items():
    assert len(d)==34,(name,len(d))
    assert d['participant_id'].nunique()==34,name
    assert 440 not in set(d['participant_id']),name
    assert set(d['label'].unique()) <= {0,1},name
    assert set(d['prediction'].unique()) <= {0,1},name

def metric_bundle(d):
    y=d['label'].to_numpy(int); p=d['probability'].to_numpy(float); q=d['prediction'].to_numpy(int)
    cm=confusion_matrix(y,q,labels=[0,1])
    cr=classification_report(y,q,labels=[0,1],target_names=['Non-depressed','Depressed'],
                             output_dict=True,zero_division=0)
    return {
        'n':int(len(y)),
        'accuracy':float(accuracy_score(y,q)),
        'balanced_accuracy':float(balanced_accuracy_score(y,q)),
        'macro_f1':float(f1_score(y,q,average='macro')),
        'depressed_f1':float(f1_score(y,q,pos_label=1,zero_division=0)),
        'auroc':float(roc_auc_score(y,p)),
        'tn':int(cm[0,0]),'fp':int(cm[0,1]),'fn':int(cm[1,0]),'tp':int(cm[1,1]),
        'confusion_matrix':cm.tolist(),
        'classification_report':cr,
    }

metrics={name:metric_bundle(d) for name,d in preds.items()}

rows=[{
    'Model':name,
    'N':m['n'],
    'Accuracy':m['accuracy'],
    'Balanced Accuracy':m['balanced_accuracy'],
    'Macro-F1':m['macro_f1'],
    'Depressed F1':m['depressed_f1'],
    'AUROC':m['auroc'],
    'TN':m['tn'],'FP':m['fp'],'FN':m['fn'],'TP':m['tp'],
} for name,m in metrics.items()]
df=pd.DataFrame(rows)
display(df.round(4))

# ---- Frozen v3 reproduction assertions ----
assert round(metrics['IDIAP text teacher']['macro_f1'],10)==round(0.8418604651162791,10)
assert round(metrics['USSD audio teacher']['macro_f1'],4)==0.7875
assert (round(metrics['v3 text branch']['macro_f1'],4),
        round(metrics['v3 text branch']['depressed_f1'],4),
        round(metrics['v3 text branch']['auroc'],4))==(0.7850,0.7407,0.8063)
assert (round(metrics['No-KD multimodal']['macro_f1'],4),
        round(metrics['No-KD multimodal']['depressed_f1'],4),
        round(metrics['No-KD multimodal']['auroc'],4))==(0.7043,0.6087,0.7470)
assert (round(metrics['Standard KD']['macro_f1'],4),
        round(metrics['Standard KD']['depressed_f1'],4),
        round(metrics['Standard KD']['auroc'],4))==(0.7236,0.6667,0.7668)

print('PASS: all frozen DEV-34 targets reproduced.')

In [ ]:
# 3) Build the complete result package in the NEW Drive run folder

OUT=RUN/'06_verification'
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'predictions').mkdir(exist_ok=True)
(OUT/'confusion_matrices').mkdir(exist_ok=True)
(OUT/'classification_reports').mkdir(exist_ok=True)
(OUT/'source_metrics').mkdir(exist_ok=True)
(OUT/'provenance').mkdir(exist_ok=True)

slug={
    'IDIAP text teacher':'idiap_text_teacher',
    'USSD audio teacher':'ussd_audio_teacher',
    'v3 text branch':'v3_text_branch',
    'No-KD multimodal':'no_kd_multimodal',
    'Standard KD':'standard_kd',
}

# Comparative table
df.to_csv(OUT/'comparative_table.csv',index=False)

def markdown_table(frame):
    cols=list(frame.columns)
    lines=['| '+' | '.join(cols)+' |','|'+'|'.join(['---']*len(cols))+'|']
    for _,r in frame.iterrows():
        vals=[]
        for c in cols:
            v=r[c]
            if isinstance(v,(float,np.floating)): vals.append(f'{float(v):.4f}')
            else: vals.append(str(v))
        lines.append('| '+' | '.join(vals)+' |')
    return '\n'.join(lines)+'\n'

(OUT/'comparative_table.md').write_text(markdown_table(df),encoding='utf-8')

# Predictions, CR, CM, and one CM image per model
all_cm={}
all_cr={}
for name,d in preds.items():
    s=slug[name]
    d.sort_values('participant_id').to_csv(OUT/'predictions'/f'{s}_dev34.csv',index=False)

    m=metrics[name]
    cm=np.asarray(m['confusion_matrix'],dtype=int)
    all_cm[name]=m['confusion_matrix']
    pd.DataFrame(cm,index=['true_0','true_1'],columns=['pred_0','pred_1']).to_csv(
        OUT/'confusion_matrices'/f'{s}.csv'
    )

    cr=m['classification_report']
    all_cr[name]=cr
    pd.DataFrame(cr).T.to_csv(OUT/'classification_reports'/f'{s}.csv')

    fig,ax=plt.subplots(figsize=(4.8,4.2))
    ConfusionMatrixDisplay(cm,display_labels=['Non-depressed','Depressed']).plot(ax=ax)
    ax.set_title(name+' - DEV-34')
    fig.tight_layout()
    fig.savefig(OUT/'confusion_matrices'/f'{s}.png',dpi=180,bbox_inches='tight')
    plt.close(fig)

(OUT/'confusion_matrices.json').write_text(json.dumps(all_cm,indent=2)+'\n')
(OUT/'classification_reports.json').write_text(json.dumps(all_cr,indent=2)+'\n')

# Preserve compact source metrics/audits used to produce the table
source_files={
    '01_text_teacher_audit.json':TTEACH/'text_target_audit.json',
    '02_audio_teacher_audit.json':ATEACH/'audit.json',
    '03_text_branch_metrics.json':TEXT/'metrics.json',
    '03_audio_branch_metrics.json':AUDIO_BRANCH/'metrics.json',
    '04_no_kd_metrics.json':NOKD/'metrics.json',
    '05_standard_kd_metrics.json':KD/'metrics.json',
}
for name,src in source_files.items():
    shutil.copy2(src,OUT/'source_metrics'/name)

# Run provenance
run_manifest=RUN/'run_manifest.json'
assert run_manifest.is_file()
shutil.copy2(run_manifest,OUT/'provenance'/'run_manifest.json')

summary={
    'status':'PASS',
    'run_id':RUN.name,
    'core_commit':CORE_COMMIT,
    'protocol':'TRAIN-107 / DEV-34; participant 440 excluded',
    'test_accessed':False,
    'threshold':0.5,
    'standard_kd':{'temperature':2.0,'kd_weight':0.5,'seed':103},
    'results':rows,
}
(OUT/'provenance'/'reproduction_summary.json').write_text(json.dumps(summary,indent=2)+'\n')

result_readme=f"""# Exact v3 reproduction result: {RUN.name}

Status: **PASS**

Protocol: TRAIN-107 / DEV-34, participant 440 excluded. TEST was not read or scored.

Core reproduction commit: `{CORE_COMMIT}`

## Result table

{markdown_table(df)}

## Included artifacts

- `comparative_table.csv` and `comparative_table.md`
- `predictions/*_dev34.csv`: participant-level DEV predictions used to recompute every metric
- `confusion_matrices/*.csv` and `*.png`
- `classification_reports/*.csv`
- `confusion_matrices.json`
- `classification_reports.json`
- `source_metrics/*.json`: original teacher/branch/fusion metric files
- `provenance/run_manifest.json`
- `provenance/reproduction_summary.json`

Large checkpoints, extracted ComParE16 features, intermediate arrays and training histories remain in the corresponding Drive run folder and are intentionally not committed to Git.
"""
(OUT/'README.md').write_text(result_readme,encoding='utf-8')

print('Result package:',OUT)
print('\n'.join(str(p.relative_to(OUT)) for p in sorted(OUT.rglob('*')) if p.is_file()))

In [ ]:
# 4) Publish the verified result package to GitHub main
#
# Creates results/<run_id>/ and refuses to overwrite an existing Git result run.

PUBLISH=Path('/content/reliability-aware-depression-kd-publish')
if PUBLISH.exists(): shutil.rmtree(PUBLISH)

subprocess.run(['git','clone','--branch','main','--single-branch',
                f'https://github.com/{REPO}.git',str(PUBLISH)],check=True,env=env)

GIT_OUT=PUBLISH/'results'/RUN.name
assert not GIT_OUT.exists(), f'Git result folder already exists; refusing overwrite: {GIT_OUT}'
GIT_OUT.parent.mkdir(parents=True,exist_ok=True)
shutil.copytree(OUT,GIT_OUT)

subprocess.run(['git','-C',str(PUBLISH),'config','user.name','exact-v3-reproduction'],check=True)
subprocess.run(['git','-C',str(PUBLISH),'config','user.email','exact-v3-reproduction@users.noreply.github.com'],check=True)
subprocess.run(['git','-C',str(PUBLISH),'add',str(Path('results')/RUN.name)],check=True)

status=subprocess.check_output(['git','-C',str(PUBLISH),'status','--porcelain'],text=True)
assert status.strip(), 'Nothing to publish.'
print(status)

subprocess.run(['git','-C',str(PUBLISH),'commit','-m',
                f'Add exact v3 reproduction results {RUN.name}'],check=True)
subprocess.run(['git','-C',str(PUBLISH),'push','origin','HEAD:main'],check=True,env=env)

published_sha=subprocess.check_output(
    ['git','-C',str(PUBLISH),'rev-parse','HEAD'],text=True
).strip()
print('================================================')
print('PASS: exact v3 DEV-34 reproduction complete')
print('Git result package:',f'results/{RUN.name}/')
print('Published commit:',published_sha)
print('TEST was not read or scored.')
print('================================================')